# Punto 1 — Fundamentos de la estadística aplicada

## *"¿Puedo confiar en estos datos?"*

**Unidad III — Estadística Descriptiva · Diplomatura en Data Analytics (UNNE)**

Esta notebook acompaña el **video del Punto 1**. No la leas de corrido: pará el video en
cada bloque y corré las celdas correspondientes.

El objetivo del punto 1 no es calcular nada todavía. Es aprender a **mirar** un dataset
antes de sacarle conclusiones: de dónde viene, qué representa cada fila, qué tipo de cosa
hay en cada columna, y dónde puede estar mintiendo.

> **Cómo está armada esta notebook.** Cada concepto viene en tres partes: primero **qué
> es**, después **dónde sirve**, y después una sección **⚠️ Forzando el concepto** donde lo
> llevamos hasta que se rompe. Esa tercera parte es la importante: entender una
> herramienta es saber dónde deja de funcionar.

---
## 0 · Setup

Una sola celda, siempre igual en todas las notebooks de la unidad.

In [19]:
import os

import pandas as pd

# Que las tablas no se corten ni usen notación científica: queremos VER los números.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# El dataset: precios de vivienda en California, censo de 1990.
# Si tenés el archivo al lado lo usa; si no, lo baja. Así corre igual en tu
# máquina y en Colab, sin cambiar nada.
RUTA = "../data/raw/housing.csv"
URL = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
       "master/datasets/housing/housing.csv")

df = pd.read_csv(RUTA) if os.path.exists(RUTA) else pd.read_csv(URL)

print(f"Cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

Cargado: 20,640 filas × 10 columnas


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.00,880.00,129.00,322.00,126.00,8.33,"452,600.00",NEAR BAY
1,-122.22,37.86,21.00,"7,099.00","1,106.00","2,401.00","1,138.00",8.30,"358,500.00",NEAR BAY
2,-122.24,37.85,52.00,"1,467.00",190.00,496.00,177.00,7.26,"352,100.00",NEAR BAY
3,-122.25,37.85,52.00,"1,274.00",235.00,558.00,219.00,5.64,"341,300.00",NEAR BAY
4,-122.25,37.85,52.00,"1,627.00",280.00,565.00,259.00,3.85,"342,200.00",NEAR BAY


---
## 1 · De dónde salen estos datos

**California Housing.** Sale del **censo de Estados Unidos de 1990** y describe el valor de
la vivienda en California. Es uno de los datasets más usados del mundo para enseñar
análisis de datos — y, como vamos a ver, viene con varios problemas reales adentro. Eso lo
hace mejor para aprender, no peor.

Toda la unidad gira alrededor de una sola pregunta, que se vuelve más exigente clase a
clase:

> ### ¿El segmento A es realmente distinto del segmento B?

Acá la versión concreta es: **¿las viviendas cerca del mar valen más que las de tierra
adentro?** Parece una pregunta simple. Vas a ver que contestarla bien lleva las cinco
clases.

| Punto | Pregunta que responde |
|---|---|
| **1 · Fundamentos** | ¿Puedo confiar en estos datos? |
| **2 · Univariada** | ¿Cómo es cada segmento por dentro? |
| **3 · Bivariada** | ¿Qué relaciona a las variables? |
| **4 · Visualización** | ¿Cómo lo cuento para que se entienda? |
| **5 · Inferencia** | ¿La diferencia es real o es azar? |

Antes de nada, las diez columnas:

| Columna | Qué dice |
|---|---|
| `longitude`, `latitude` | dónde queda, en coordenadas |
| `housing_median_age` | antigüedad mediana de las viviendas |
| `total_rooms`, `total_bedrooms` | cantidad de ambientes y de dormitorios |
| `population`, `households` | cuánta gente y cuántos hogares |
| `median_income` | ingreso mediano |
| `median_house_value` | **valor mediano de la vivienda** — la variable protagonista |
| `ocean_proximity` | cercanía al mar — **la categórica que define los segmentos** |

### Lo primero, y lo que más errores evita: ¿qué es una fila?

Antes de calcular nada hay que saber **qué representa un renglón**. Es la pregunta que casi
nadie se hace.

In [20]:
df.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.00,880.00,129.00,322.00,126.00,8.33,"452,600.00",NEAR BAY
1,-122.22,37.86,21.00,"7,099.00","1,106.00","2,401.00","1,138.00",8.30,"358,500.00",NEAR BAY
2,-122.24,37.85,52.00,"1,467.00",190.00,496.00,177.00,7.26,"352,100.00",NEAR BAY


Mirá `total_rooms` en la primera fila: **880 ambientes**. Y `population`: **322 personas**.

Una fila **no es una casa**. Es un ***block group***: la unidad más chica que publica el
censo de EE.UU., que agrupa entre 600 y 3.000 personas. Todos los números de la fila son
**agregados de ese barrio**, no de una vivienda.

In [21]:
print(f"Población por fila:  media = {df['population'].mean():,.0f}"
      f"   mediana = {df['population'].median():,.0f}")
print(f"Hogares por fila:    media = {df['households'].mean():,.0f}")
print(f"Ambientes por fila:  media = {df['total_rooms'].mean():,.0f}"
      f"   máximo = {df['total_rooms'].max():,.0f}")
print()
print("39.320 ambientes no es una mansión: es un barrio entero sumado.")

Población por fila:  media = 1,425   mediana = 1,166
Hogares por fila:    media = 500
Ambientes por fila:  media = 2,636   máximo = 39,320

39.320 ambientes no es una mansión: es un barrio entero sumado.


Entonces, `total_rooms` no se puede interpretar solo. Hay que **dividirlo por la cantidad
de hogares** para que signifique algo:

In [22]:
ambientes_por_hogar = df["total_rooms"] / df["households"]

print(f"Ambientes por hogar:  media = {ambientes_por_hogar.mean():.2f}"
      f"   mediana = {ambientes_por_hogar.median():.2f}")
print()
print("~5 ambientes por hogar. ESE número es interpretable; 2.636 no lo era.")

Ambientes por hogar:  media = 5.43   mediana = 5.23

~5 ambientes por hogar. ESE número es interpretable; 2.636 no lo era.


---
## 2 · Estadística descriptiva vs. inferencia

Dos objetivos distintos sobre los mismos datos.

| | **Descriptiva** | **Inferencia** |
|---|---|---|
| **Qué hace** | describe y resume lo que tenés en la mano | saca conclusiones sobre algo más grande que lo que medís |
| **Ejemplo acá** | "estos barrios cerca del mar valieron en promedio X" | "¿las viviendas cerca del mar valen más *en general*?" |
| **Cuándo se equivoca** | casi nunca: es un cálculo sobre datos que tenés | seguido: estás afirmando algo que no medís del todo |
| **En la unidad** | puntos 1 a 4 | punto 5 |

La diferencia práctica es el **verbo**. "Valieron" es descripción. "Valen" es inferencia:
estás hablando también de las viviendas y los años que no mediste.

In [23]:
# DESCRIPCIÓN: un hecho sobre las filas que tengo. No se discute.
medias = (df.groupby("ocean_proximity")["median_house_value"]
            .mean().sort_values(ascending=False))
print("Valor mediano promedio por cercanía al mar (USD, censo 1990):\n")
print(medias.round(0))

# INFERENCIA sería decir: "las casas cerca del mar valen más".
# Eso ya no es un cálculo, es una afirmación sobre el mundo.
# Y fijate en ISLAND, arriba de todo: ¿cuántos datos tendrá? Volvemos a eso.

Valor mediano promedio por cercanía al mar (USD, censo 1990):

ocean_proximity
ISLAND       380,440.00
NEAR BAY     259,212.00
NEAR OCEAN   249,434.00
<1H OCEAN    240,084.00
INLAND       124,805.00
Name: median_house_value, dtype: float64


---
## 3 · Tipos de variable

Del tipo de cada variable dependen las operaciones y los gráficos que **tienen sentido**.

Hay dos preguntas, en este orden:

**1. ¿Es una categoría o una cantidad?**

- **Cualitativa (categórica)** → nombra o clasifica.
  - **Nominal**: sin orden. `ocean_proximity`.
  - **Ordinal**: con orden, pero sin distancias comparables.
- **Cuantitativa (numérica)** → mide, y la aritmética tiene sentido.
  - **Discreta**: se cuenta, valores separados. `households`, `total_rooms`.
  - **Continua**: se mide, admite decimales. `median_income`, `median_house_value`.

**2. Si es cuantitativa, ¿el cero significa "nada"?** Importa para el punto 2: sólo si el
cero es absoluto tiene sentido decir "el doble".

In [24]:
# Miremos qué tipo de dato leyó pandas para cada columna.
# OJO: el dtype es lo que pandas ADIVINÓ, no lo que la variable ES.
tipos = pd.DataFrame({
    "dtype_pandas": df.dtypes.astype(str),
    "valores_distintos": df.nunique(),
    "ejemplo": [df[c].dropna().iloc[0] for c in df.columns],
})
tipos

,dtype_pandas,valores_distintos,ejemplo
longitude,float64,844,-122.23
latitude,float64,862,37.88
housing_median_age,float64,52,41.00
total_rooms,float64,5926,880.00
total_bedrooms,float64,1923,129.00
population,float64,3888,322.00
households,float64,1815,126.00
median_income,float64,12928,8.33
median_house_value,float64,3842,"452,600.00"
ocean_proximity,object,5,NEAR BAY


#### ⚠️ Forzando el concepto: cuatro columnas donde el `dtype` se queda corto

El `dtype` de pandas sólo distingue *cómo está guardado el dato*, no *qué significa*. Estas
cuatro son las que hay que discutir:

In [25]:
# ── 1. longitude / latitude: son float64, pero ¿son cantidades?
print("longitude / latitude  →  dtype:", df["longitude"].dtype)
print(f"     media = ({df['longitude'].mean():.2f}, {df['latitude'].mean():.2f})")
print("     Es un punto en el mapa: el centro geográfico de los barrios medidos.")
print("     El cálculo es válido, pero 'la longitud promedio' no describe nada")
print("     que alguien vaya a preguntar. Y las dos SÓLO significan algo juntas.\n")

# ── 2. ocean_proximity: texto. ¿Nominal u ordinal?
print("ocean_proximity  →  categorías:", list(df["ocean_proximity"].unique()))
print("     Cuatro de ellas insinúan un orden por distancia al mar:")
print("     NEAR BAY < NEAR OCEAN < <1H OCEAN < INLAND.")
print("     Pero ISLAND rompe la escala, y '<1H OCEAN' mide TIEMPO DE VIAJE,")
print("     no distancia. ¿Es ordinal? Es discutible, y hay que decidirlo.\n")

# ── 3. housing_median_age: continua... hasta que mirás el máximo.
print("housing_median_age  →  dtype:", df["housing_median_age"].dtype)
print(f"     min = {df['housing_median_age'].min():.0f}   max = {df['housing_median_age'].max():.0f}")
print(f"     filas exactamente en el máximo: {(df['housing_median_age'] == 52).sum():,}")
print("     ¿En California no hay ninguna casa de más de 52 años? Claro que sí.")
print("     El 52 es un TOPE: todo lo más viejo quedó aplastado ahí.\n")

# ── 4. median_income: el nombre engaña.
print("median_income  →  dtype:", df["median_income"].dtype)
print(f"     min = {df['median_income'].min():.4f}   max = {df['median_income'].max():.4f}")
print("     Nadie tiene un ingreso de 3,87. Está en DECENAS DE MILES de dólares,")
print("     y también viene topeado arriba (15,0001) y abajo (0,4999).")

longitude / latitude  →  dtype: float64
     media = (-119.57, 35.63)
     Es un punto en el mapa: el centro geográfico de los barrios medidos.
     El cálculo es válido, pero 'la longitud promedio' no describe nada
     que alguien vaya a preguntar. Y las dos SÓLO significan algo juntas.

ocean_proximity  →  categorías: ['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
     Cuatro de ellas insinúan un orden por distancia al mar:
     NEAR BAY < NEAR OCEAN < <1H OCEAN < INLAND.
     Pero ISLAND rompe la escala, y '<1H OCEAN' mide TIEMPO DE VIAJE,
     no distancia. ¿Es ordinal? Es discutible, y hay que decidirlo.

housing_median_age  →  dtype: float64
     min = 1   max = 52
     filas exactamente en el máximo: 1,273
     ¿En California no hay ninguna casa de más de 52 años? Claro que sí.
     El 52 es un TOPE: todo lo más viejo quedó aplastado ahí.

median_income  →  dtype: float64
     min = 0.4999   max = 15.0001
     Nadie tiene un ingreso de 3,87. Está en DECENAS DE MILE

**El resumen de lo anterior:**

| Columna | Lo que dice pandas | Lo que realmente es | El problema |
|---|---|---|---|
| `longitude`/`latitude` | `float64` | coordenadas, sólo con sentido en par | promediarlas casi nunca sirve |
| `ocean_proximity` | `object` | nominal… ¿u ordinal? | hay que decidirlo a mano |
| `housing_median_age` | `float64` | continua **topeada en 52** | el tope deforma todo lo que calcules |
| `median_income` | `float64` | continua **escalada y topeada** | la unidad no es la que parece |

Fijate que la conclusión no se deduce del `dtype` en ningún caso.

---
## 4 · Mirar un dataset por primera vez

Tres funciones de pandas, tres preguntas distintas. Ninguna reemplaza a las otras.

### 4.1 · `.info()` → ¿qué tipos hay y cuántos datos faltan?

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


Lo que hay que leer acá, en orden:

1. **`Non-Null Count`** — dónde faltan datos. `total_bedrooms` tiene 20.433 de 20.640:
   **faltan 207**. Es el único faltante declarado del dataset.
2. **`Dtype`** — si una columna que debería ser numérica aparece como `object`, pandas la
   leyó como texto y **no vas a poder calcular nada con ella**.
3. **`memory usage`** — irrelevante con 20.000 filas, importante con millones.

#### ⚠️ Forzando el concepto: lo que `.info()` no puede ver

`.info()` cuenta los `NaN`. Pero un dato puede estar **mal sin ser NaN**:

In [27]:
# .info() dice que median_house_value está completa. Y es cierto: no hay ningún NaN.
print("Nulos en median_house_value:", df["median_house_value"].isna().sum())

# Pero mirá el valor más alto y cuántas filas lo tienen:
tope = df["median_house_value"].max()
n_tope = (df["median_house_value"] == tope).sum()

print(f"Valor máximo                : {tope:,.0f}")
print(f"Filas exactamente en ese valor: {n_tope:,}  ({n_tope/len(df):.1%} del dataset)")
print()
print("Si fuera un valor real, sería una coincidencia imposible: 965 barrios")
print("distintos de toda California valiendo exactamente 500.001 dólares.")
print("No es un valor: es un TOPE. Todo lo que valía más quedó aplastado ahí.")

Nulos en median_house_value: 0
Valor máximo                : 500,001
Filas exactamente en ese valor: 965  (4.7% del dataset)

Si fuera un valor real, sería una coincidencia imposible: 965 barrios
distintos de toda California valiendo exactamente 500.001 dólares.
No es un valor: es un TOPE. Todo lo que valía más quedó aplastado ahí.


### 4.2 · `.describe()` → ¿los números son razonables?

In [28]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,"20,640.00","20,640.00","20,640.00","20,640.00","20,433.00","20,640.00","20,640.00","20,640.00","20,640.00"
mean,-119.57,35.63,28.64,"2,635.76",537.87,"1,425.48",499.54,3.87,"206,855.82"
std,2.00,2.14,12.59,"2,181.62",421.39,"1,132.46",382.33,1.90,"115,395.62"
min,-124.35,32.54,1.00,2.00,1.00,3.00,1.00,0.50,"14,999.00"
25%,-121.80,33.93,18.00,"1,447.75",296.00,787.00,280.00,2.56,"119,600.00"
50%,-118.49,34.26,29.00,"2,127.00",435.00,"1,166.00",409.00,3.53,"179,700.00"
75%,-118.01,37.71,37.00,"3,148.00",647.00,"1,725.00",605.00,4.74,"264,725.00"
max,-114.31,41.95,52.00,"39,320.00","6,445.00","35,682.00","6,082.00",15.00,"500,001.00"


Cómo se lee, en orden de importancia:

1. **`min` y `max`** primero. ¿Hay algo imposible? Acá el `max` de `median_house_value` es
   exactamente 500.001, que es justo lo que acabamos de descubrir.
2. **`mean` contra `50%` (la mediana).** Si se parecen, la distribución es simétrica. Si la
   media es mayor, hay valores grandes tirando de ella.
3. **`std`** — cuánto se dispersan los datos alrededor de la media.

#### ⚠️ Forzando el concepto (1): `.describe()` te ocultó una columna entera

In [29]:
print("Columnas del dataset      :", df.shape[1])
print("Columnas que muestra describe():", df.describe().shape[1])
print()
print("Falta ocean_proximity — justo LA categórica que define los segmentos,")
print("la variable más importante para nuestra pregunta. describe() la omite")
print("en silencio porque no es numérica.")

Columnas del dataset      : 10
Columnas que muestra describe(): 9

Falta ocean_proximity — justo LA categórica que define los segmentos,
la variable más importante para nuestra pregunta. describe() la omite
en silencio porque no es numérica.


In [30]:
# Para las categóricas hay que pedirlo explícitamente:
df["ocean_proximity"].describe()

count         20640
unique            5
top       <1H OCEAN
freq           9136
Name: ocean_proximity, dtype: object

#### ⚠️ Forzando el concepto (2): `.describe()` calcula sin entender

In [31]:
# describe() promedia TODA columna numérica, tenga sentido o no.
df[["longitude", "latitude"]].describe()

,longitude,latitude
count,"20,640.00","20,640.00"
mean,-119.57,35.63
std,2.00,2.14
min,-124.35,32.54
25%,-121.80,33.93
50%,-118.49,34.26
75%,-118.01,37.71
max,-114.31,41.95


> **La regla:** `.describe()` es un punto de partida para *mirar*, nunca un resultado para
> *reportar*. Decide por `dtype`, y el `dtype` no sabe qué mide la columna.

### 4.3 · `.value_counts()` → ¿cómo se reparten las categorías?

In [32]:
# Para una categórica es la herramienta correcta: cuántos casos por categoría.
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

Y acá aparece algo importante para el resto de la unidad: **`ISLAND` tiene 5 filas**.

Cinco, contra 9.136 de `<1H OCEAN`. Sin embargo, si volvés a la tabla de promedios de la
sección 2, `ISLAND` era **el grupo con el valor promedio más alto de todos**.

¿Las casas en islas son las más caras de California?

#### ⚠️ Forzando el concepto: la herramienta equivocada que encuentra el problema

In [33]:
# value_counts() sobre una variable CONTINUA no debería servir para nada:
# si cada valor es único, todos los conteos son 1.
df["median_house_value"].value_counts().head(5)

median_house_value
500,001.00    965
137,500.00    122
162,500.00    117
112,500.00    103
187,500.00     93
Name: count, dtype: int64

Y sin embargo ahí está: **500.001 aparece 965 veces**. Un valor exacto repetido cientos de
veces en una variable continua es casi siempre la señal de un tope o de un relleno.

> Usamos una herramienta "donde no correspondía" y encontramos el problema más importante
> del dataset. Por eso la exploración inicial se hace sin prejuicios: mirá todo de todas las
> formas antes de decidir qué es serio.

---
## 5 · Por qué un tope no es un detalle

Podría parecer un problema menor: son 965 filas sobre 20.640, menos del 5%. Veamos cuánto
cambia lo que calculás.

In [34]:
tope = df["median_house_value"].max()
sin_tope = df[df["median_house_value"] < tope]["median_house_value"]

print(f"Media de TODAS las filas        : {df['median_house_value'].mean():>10,.0f}")
print(f"Media sin las 965 topeadas      : {sin_tope.mean():>10,.0f}")
print(f"                     diferencia : {df['median_house_value'].mean() - sin_tope.mean():>10,.0f} USD")
print()
print("Las topeadas empujan la media general 14.378 dólares hacia arriba,")
print("lo cual tiene sentido: son las caras.")
print()
print("Pero acá está lo importante: esas 965 filas NO valían 500.001.")
print("Valían MÁS, y no sabemos cuánto. El tope las recortó hacia abajo.")
print()
print("O sea que 206.856 no es 'el promedio de California en 1990':")
print("es un PISO. El valor verdadero es más alto, y no se puede recuperar")
print("del dataset. Eso ya no se arregla con estadística.")

Media de TODAS las filas        :    206,856
Media sin las 965 topeadas      :    192,478
                     diferencia :     14,378 USD

Las topeadas empujan la media general 14.378 dólares hacia arriba,
lo cual tiene sentido: son las caras.

Pero acá está lo importante: esas 965 filas NO valían 500.001.
Valían MÁS, y no sabemos cuánto. El tope las recortó hacia abajo.

O sea que 206.856 no es 'el promedio de California en 1990':
es un PISO. El valor verdadero es más alto, y no se puede recuperar
del dataset. Eso ya no se arregla con estadística.


#### ⚠️ Forzando el concepto: el tope no está repartido parejo

Acá está la parte que realmente importa para nuestra pregunta. Si el tope afectara a todos
los grupos por igual, al compararlos se cancelaría. Pero mirá:

In [35]:
topeadas = df[df["median_house_value"] == tope]

comparacion = pd.DataFrame({
    "filas": df["ocean_proximity"].value_counts(),
    "topeadas": topeadas["ocean_proximity"].value_counts(),
}).fillna(0)
comparacion["% topeadas"] = comparacion["topeadas"] / comparacion["filas"] * 100
comparacion.sort_values("% topeadas", ascending=False).round(1)

,filas,topeadas,% topeadas
ocean_proximity,,,
NEAR BAY,2290,194.00,8.50
NEAR OCEAN,2658,212.00,8.00
<1H OCEAN,9136,532.00,5.80
INLAND,6551,27.00,0.40
ISLAND,5,0.00,0.00


**El tope golpea muy distinto a cada segmento.** Los barrios cerca del mar tienen entre **14 y
21 veces** más proporción de filas topeadas que los de tierra adentro (`INLAND`, 0,4%).

Como el tope recorta justamente los valores más altos, y los valores más altos están cerca
del mar, **la diferencia real entre los grupos es todavía mayor de la que muestran los
datos**. El dataset subestima la brecha, y la subestima de forma desigual.

> Un problema de calidad no es sólo "ruido": puede sesgar sistemáticamente la comparación
> que justamente querés hacer. Por eso se audita **antes** de comparar, no después.

#### ⚠️ Forzando el concepto: los defectos también se propagan

In [36]:
# Antes dividimos total_rooms por households para que el número fuera interpretable.
# Miremos los casos extremos de esa variable derivada:
derivada = df.assign(ambientes_por_hogar=df["total_rooms"] / df["households"])

derivada.nlargest(3, "ambientes_por_hogar")[
    ["total_rooms", "households", "population", "ambientes_por_hogar", "median_house_value"]
]

,total_rooms,households,population,ambientes_por_hogar,median_house_value
1914,"1,561.00",11.00,30.00,141.91,"500,001.00"
1979,"1,988.00",15.00,36.00,132.53,"162,500.00"
12447,"2,809.00",45.00,83.00,62.42,"87,500.00"


**142 ambientes por hogar.** Un barrio con 1.561 ambientes, 30 personas y 11 hogares.

Eso no es una vivienda familiar: parece un edificio institucional —una residencia, un
hotel, algo por el estilo— contado como si fuera un barrio residencial. Y fijate que esa
misma fila **también está topeada** en 500.001.

Nadie tocó `ambientes_por_hogar`: la calculamos nosotros hace tres celdas. El problema
estaba en los datos de origen y **viajó solo** a la variable nueva.

> Si no auditás al principio, el problema no desaparece: se disfraza. Aparece más adelante
> como un número raro en una columna que vos mismo creaste, cuando ya nadie se acuerda de
> dónde salió.

---
## 6 · Lo que hay que llevarse del punto 1

1. **Antes de calcular, entender.** Qué representa una fila, de dónde salieron los datos,
   quién los midió y con qué definición. Acá, una fila es un barrio, no una casa.
2. **El tipo de variable lo decidís vos, no el `dtype`.** Pandas guarda; vos interpretás.
3. **Un dato puede estar mal sin ser `NaN`.** Topes, valores de relleno, escalas ocultas.
4. **Un problema de calidad puede sesgar la comparación**, no sólo agregar ruido.
5. **Los defectos se propagan** a todo lo que calcules a partir de ellos.